# 09 — Backbone cho paper (Bảng 4) trên Google Colab A100 · DOWN

Chạy `experiments/stage7_backbones.yaml`: **8 backbone × 2 nhánh**, fold 0, cùng một công thức — chỉ backbone thay đổi.

| Nhánh | Mô hình |
|---|---|
| `M0` (full phương pháp) | MoE hướng D · d_model 256 · 3 head long-tail · clin + derm + metadata · r5_nobce |
| `img` (chỉ backbone) | backbone dùng chung 2 ảnh → nối đặc trưng → MLP 512 → 11 lớp · CE + sampler q0.5 · không metadata |

Backbone: ResNet-50, ResNet-152, EfficientNetV2-M, ConvNeXt-B, ViT-B/16, DINOv2-B/14 (392px), Swin-B, Swin-L — 384px.
Chung: lr 5e-5 (backbone ×1.0), drop path 0.1, dropout 0.4, aug mạnh, 20 epoch, patience 5, batch hiệu dụng 64.
Bảng 4: Dice chỉ backbone, Dice full và **Δ = full − chỉ backbone** cho từng backbone.

**Thư mục riêng trên Drive:** `MyDrive/milk10k_backbones/outputs` (tách khỏi `milk10k_ablation` của notebook 08).
**Phần đã có được bỏ qua:** `swin_base_384__abl_M0__f0` (M0 của notebook 08) được chép từ `milk10k_ablation/outputs`,
không train lại → còn 15 run.

**Cách chạy**
1. Runtime → Change runtime type → **A100 GPU**, bật **High-RAM**.
2. Runtime → **Run all**, cho phép truy cập Google Drive.
3. Colab ngắt giữa chừng: mở lại và **Run all**. Run đã xong (đã đồng bộ lên Drive) được bỏ qua, run dở dang chạy lại.
4. Kết quả: `MyDrive/milk10k_backbones/outputs/ablation_tables/ablation_results.xlsx` — gộp cả run của notebook 08,
   nên file Excel này có đủ 5 bảng (Bảng 4 mới).

**Notebook DOWN — chạy từ DƯỚI LÊN** (ResNet-50 → … → Swin-L).
Dùng cùng lúc với [`09_backbones_colab_top`](https://colab.research.google.com/github/trong5nhan6/fussion/blob/main/notebooks/09_backbones_colab_top.ipynb)
trên máy thứ hai: hai notebook dùng chung `_claims/` trong thư mục Drive mới, gặp nhau ở giữa, không chạy trùng.
Chạy 1 mình cũng được: notebook này tự chạy hết (theo thứ tự ngược).

In [ ]:
# ===================== CẤU HÌNH =====================
REPO_URL = "https://github.com/trong5nhan6/fussion.git"
BRANCH = "main"
DRIVE_URL = "https://drive.google.com/file/d/1lM1nB_4pq3Vg-GwlaTMQjtus_bpbMhGX/view?usp=drive_link"  # zip dữ liệu MILK10k
DRIVE_DIR = "/content/drive/MyDrive/milk10k_backbones"     # thư mục MỚI cho notebook này
ABLATION_DIR = "/content/drive/MyDrive/milk10k_ablation"   # kết quả notebook 08: lấy run đã có + gộp vào Excel

# ===================== CHỌN THÍ NGHIỆM =====================
EXPERIMENT = "experiments/stage7_backbones.yaml"
BACKBONES = None         # None = cả 8 | vd ["convnext_base_384", "resnet50_384"]
VARIANTS = ["M0", "img"] # M0 = full phương pháp | img = chỉ backbone

HUONG = "down"          # down: chạy danh sách run từ dưới lên (notebook này) | notebook top chạy từ trên xuống
TIME_BUDGET_H = 23.0     # không bắt đầu run mới nếu không kịp xong trước mốc này (phiên Colab tối đa ~24 giờ)
JOBS_PER_GPU = "auto"    # auto: 3 nếu VRAM >= 70GB, 1 nếu 40GB | hoặc số cụ thể (2 nếu bị CUDA out of memory)

# (Tuỳ chọn) run backbone cũ của notebook 07 (Kaggle) -> sheet tham khảo 4b_Backbone_Kaggle
REFERENCE = [ABLATION_DIR + "/nb07_outputs"]
# Dice test leaderboard của các file đã nộp -> điền vào bảng. Khoá = tên run hoặc tên thư mục trong submissions/
TEST_SCORES = {
    "swin_base_384__D__r5_nobce__clin+derm": 0.512,
    # "convnext_base_384__abl_M0__f0": None,
}

In [ ]:
# ===================== THAM SỐ TRAIN =====================
# Batch lấy từ configs/env/colab_a100.yaml (16 x 4, không grad checkpointing); Swin-L, EfficientNetV2-M, ResNet-152
# dùng 8 x 8 (đặt trong yaml). Không đặt batch ở đây: PARAMS ghi đè mọi backbone.
PARAMS = {
    "train.amp_dtype": "bf16",
    "train.epochs": 20,
    "train.patience": 5,
    "seed": 42,
    "predict.tta": True,
}
RUN_SUFFIX = ""

## 1. Google Drive, GPU và cấu hình song song

In [ ]:
import os, subprocess, torch
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_DIR + "/outputs", exist_ok=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime -> Change runtime type -> A100 GPU"
props = torch.cuda.get_device_properties(0)
vram = props.total_memory / 2**30
if "A100" not in props.name:
    print(f"CẢNH BÁO: đang dùng {props.name}, không phải A100. bf16 cần GPU Ampere trở lên (A100, L4, H100).")
jobs = JOBS_PER_GPU if JOBS_PER_GPU != "auto" else (3 if vram >= 70 else 1)
cpus = os.cpu_count()
PARAMS["data.num_workers"] = max(2, cpus // jobs - 1)
GPUS = ",".join(["0"] * jobs)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # giảm phân mảnh VRAM
print(f"{props.name} {vram:.0f}GB | {cpus} CPU | {jobs} run song song | {PARAMS['data.num_workers']} worker/run")

## 2. Code và dữ liệu

In [ ]:
REPO_DIR = "/content/" + REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
DATA_DIR = "/content/MILK10k"            # ổ SSD của Colab: đọc ảnh nhanh hơn Drive
os.environ["MILK10K_ROOT"] = DATA_DIR
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!pip install -q "timm>=1.0" gdown openpyxl
!git log -1 --format="Commit: %h %s"
!python scripts/setup_data.py --drive-url "{DRIVE_URL}" --dest {DATA_DIR}

In [ ]:
# Kiểm tra nhanh toàn bộ pipeline (~2 phút)
!python -m pytest tests -q -p no:cacheprovider

## 3. Đồng bộ kết quả lên Drive và danh sách run

In [ ]:
# Chạy tiếp: run đã xong trên Drive mới được chép về outputs/ và bỏ qua. Từ thư mục của notebook 08 chỉ chép các run
# thuộc lưới này (swin_base_384__abl_M0__f0), không chép 32 run ablation.
# --shared-dir: 2 notebook (top/down) không chạy trùng run.
restore_arg = (f"--restore {DRIVE_DIR}/outputs {ABLATION_DIR}/outputs --shared-dir {DRIVE_DIR}/outputs"
               + (" --reverse" if HUONG == "down" else ""))
GPU_LOG = f"outputs/_gpu_usage_{HUONG}.csv"
# Đồng bộ nền mỗi 5 phút (bỏ checkpoint .pt) + ghi mức dùng GPU mỗi phút vào GPU_LOG
os.makedirs("outputs", exist_ok=True)
sync_cmd = f"while true; do rsync -a --exclude '*.pt' outputs/ '{DRIVE_DIR}/outputs/'; sleep 300; done"
if "sync_proc" not in globals() or sync_proc.poll() is not None:
    sync_proc = subprocess.Popen(["bash", "-c", sync_cmd])
    gpu_log = subprocess.Popen("nvidia-smi --query-gpu=timestamp,utilization.gpu,memory.used,power.draw "
                               f"--format=csv -l 60 > {GPU_LOG}", shell=True)
print("Đồng bộ nền:", DRIVE_DIR + "/outputs")

In [ ]:
import shlex, yaml

def to_set_args(params: dict) -> str:
    # Chuyển PARAMS thành chuỗi --set key=value cho scripts (bỏ qua giá trị None)
    items = []
    for k, v in params.items():
        if v is None:
            continue
        val = yaml.safe_dump(v, default_flow_style=True).strip().removesuffix("...").strip()
        items.append(shlex.quote(f"{k}={val}"))
    return ("--set " + " ".join(items)) if items else ""

set_arg = to_set_args(PARAMS)
suffix_arg = f"--suffix {RUN_SUFFIX}" if RUN_SUFFIX else ""
print("Override:", set_arg or "(không có)")
print("Hậu tố tên run:", RUN_SUFFIX or "(không có)")

In [ ]:
import yaml
exp = yaml.safe_load(open(EXPERIMENT, encoding="utf-8"))
select_arg = f"--variants {','.join(VARIANTS)}" + (f" --backbones {','.join(BACKBONES)}" if BACKBONES else "")

# Phút / run trên A100 khi chạy 1 mình (ước lượng); nhiều run cùng GPU thì mỗi run chậm hơn
MIN = {"swin_large_384": 120, "dinov2_base_392": 80, "convnext_base_384": 70, "swin_base_384": 45,
       "vit_base_384": 50, "efficientnetv2_m": 50, "resnet152_384": 45, "resnet50_384": 25}
bbs = BACKBONES or list(exp["backbones"])
total = sum(MIN.get(b, 60) for b in bbs) * len(VARIANTS)
speed = {1: 1.0, 2: 1.6, 3: 1.8}.get(jobs, 1.8)
print(f"{len(bbs) * len(VARIANTS)} run (trừ run đã có), ước tính ~{total / 60 / speed:.1f} giờ trên A100 "
      f"({jobs} run song song). Số đo thật in ra sau run đầu tiên.")
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --dry-run

## 4. Train

In [ ]:
!python scripts/run_grid.py {EXPERIMENT} --env configs/env/colab_a100.yaml {select_arg} {set_arg} {suffix_arg} {restore_arg} --gpus {GPUS} --est-run-h 1.0 --time-budget-h {TIME_BUDGET_H}

## 5. Bảng kết quả và file Excel
Gộp run của notebook này **và** của notebook 08 (`ABLATION_DIR`): đủ 5 bảng, Bảng 4 = 8 backbone × (chỉ backbone | full).
Chạy 2 máy: đợi cả hai xong rồi chạy lại ô này ở một máy. Chỉ chạy khi log Train báo đã xong toàn bộ lưới.
Nộp file lên leaderboard, điền điểm vào `TEST_SCORES` rồi **chỉ chạy lại ô này**.

In [ ]:
from IPython.display import Markdown, display
!rsync -a --exclude '*.pt' outputs/ "{DRIVE_DIR}/outputs/"
yaml.safe_dump({k: v for k, v in TEST_SCORES.items() if v is not None}, open("outputs/test_scores.yaml", "w"))
ref_dirs = " ".join(f'"{r}"' for r in [ABLATION_DIR + "/outputs", *REFERENCE] if os.path.isdir(r))
!python scripts/ablation_tables.py --dirs "{DRIVE_DIR}/outputs" {ref_dirs} --test-scores outputs/test_scores.yaml > /dev/null
display(Markdown(open("outputs/ablation_tables/ablation_tables.md", encoding="utf-8").read()))
!rsync -a outputs/ablation_tables/ "{DRIVE_DIR}/outputs/ablation_tables/"
print("Excel:", DRIVE_DIR + "/outputs/ablation_tables/ablation_results.xlsx")

In [ ]:
# Mức dùng GPU trong lúc train (lấy mẫu mỗi phút)
import pandas as pd
u = pd.read_csv(GPU_LOG, skipinitialspace=True)
util = u.filter(like="utilization").iloc[:, 0].str.rstrip(" %").astype(float)
mem = u.filter(like="memory.used").iloc[:, 0].str.rstrip(" MiB").astype(float) / 1024
print(f"GPU util trung bình {util.mean():.0f}% (trung vị {util.median():.0f}%), VRAM tối đa {mem.max():.1f}GB, {len(u)} mẫu")